# BEARS Treatment Analysis & Dashboard Generation

This notebook implements an end-to-end pipeline for the BEARS aphasia treatment dataset. It performs data cleaning, preprocessing, aggregation, and generates per-player dashboards to visualize treatment progress.

## Data Sources
1. **Coins & Stars (`5. 2019-12-3_coins-stars.csv`)**: Contains gamification rewards (coins, stars) per category block.
2. **Naming Data (`2. 2019-10-25_treatment_retrieval_noprime.csv`)**: Contains trial-level naming performance metrics (accuracy, reaction time).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# File paths
COINS_STARS_FILE = '5. 2019-12-3_coins-stars.csv'
NAMING_DATA_FILE = '2. 2019-10-25_treatment_retrieval_noprime.csv'

print("Libraries loaded.")

## 1. Load Data
We load the rewards and naming datasets into pandas DataFrames.

In [ ]:
def load_data():
    try:
        rewards_df = pd.read_csv(COINS_STARS_FILE)
        naming_df = pd.read_csv(NAMING_DATA_FILE)
        print(f"Loaded {len(rewards_df)} rows of rewards data and {len(naming_df)} rows of naming data.")
        return rewards_df, naming_df
    except FileNotFoundError as e:
        print(f"Error: File not found. {e}")
        return None, None

rewards_raw, naming_raw = load_data()

## 2. Cleaning & Preprocessing Functions
We define logic for cleaning naming trials (dropping invalid responses, handling reaction times) and winsorizing speed data to minimize the impact of outliers.

In [ ]:
def clean_naming_data(df):
    # Standardize player name
    df = df.copy()
    df['player'] = df['player'].str.lower()
    df['session'] = pd.to_numeric(df['session'], errors='coerce')
    
    # Keep only confirmed trials (keys == 1)
    df['naming1_confirmation_resp.keys'] = pd.to_numeric(df['naming1_confirmation_resp.keys'], errors='coerce')
    df = df[df['naming1_confirmation_resp.keys'] == 1]
    
    # Drop rows with missing naming results
    df = df.dropna(subset=['naming1_resp.corr', 'naming1_vocal.rt'])
    
    # RT Cleaning: remove RT <= 0
    df = df[df['naming1_vocal.rt'] > 0]
    
    # Winsorize RT at 99th percentile per trial (clinical standard)
    rt_99 = df['naming1_vocal.rt'].quantile(0.99)
    df['rt_winsor'] = df['naming1_vocal.rt'].clip(upper=rt_99)
    
    return df

def clean_reward_data(df):
    df = df.copy()
    df['player'] = df['player'].str.lower()
    df['session'] = pd.to_numeric(df['session'], errors='coerce')
    return df

## 3. Aggregation Logic
We aggregate both datasets by player and session, then calculate the timepoint index (1..T) for each player based on sessions where naming data exists.

In [ ]:
def process_and_merge(rewards_df, naming_df):
    # Clean
    rewards = clean_reward_data(rewards_df)
    naming = clean_naming_data(naming_df)
    
    # Aggregate Rewards
    reward_agg = rewards.groupby(['player', 'session']).agg({
        'coins': 'mean',
        'stars': 'mean',
        'category': 'count'
    }).rename(columns={
        'coins': 'coins_mean',
        'stars': 'stars_mean',
        'category': 'n_blocks'
    }).reset_index()
    
    # Aggregate Naming
    naming_agg = naming.groupby(['player', 'session']).agg({
        'naming1_resp.corr': 'mean',
        'rt_winsor': 'median',
        'block_number': 'count'
    }).rename(columns={
        'naming1_resp.corr': 'acc_mean',
        'rt_winsor': 'rt_median',
        'block_number': 'n_trials'
    }).reset_index()
    
    # Merge - keep sessions ONLY if naming data exists
    merged = pd.merge(naming_agg, reward_agg, on=['player', 'session'], how='left')
    
    # Assign Timepoint Index (1..T) per player
    merged = merged.sort_values(['player', 'session'])
    merged['timepoint'] = merged.groupby('player').cumcount() + 1
    
    return merged

summary_data = process_and_merge(rewards_raw, naming_raw)

## 4. Plotting & Export Loop
We iterate through each player to generate a unified 3-panel dashboard. 
The panels include:
1. **Efficiency**: Mean Coins and Stars with rolling averages.
2. **Accuracy**: Treatment naming accuracy (Attempt 1).
3. **Reaction Time**: Treatment naming speed (Median RT).

Outputs are saved to the `./dashboards` directory.

In [ ]:
def create_dashboards(df):
    output_dir = './dashboards'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    players = df['player'].unique()
    
    for player in players:
        p_df = df[df['player'] == player].sort_values('timepoint')
        
        # Save CSV summary for this player
        p_df.to_csv(f"{output_dir}/player_{player}_clean_summary.csv", index=False)
        
        fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
        
        # Shared Rolling Mean Settings
        window = 3
        
        # --- Panel 1: Efficiency ---
        # Coins
        raw_coins = p_df['coins_mean']
        roll_coins = raw_coins.rolling(window=window, min_periods=1).mean()
        axes[0].plot(p_df['timepoint'], raw_coins, color='#ff7f0e', alpha=0.3, marker='o', markersize=4)
        axes[0].plot(p_df['timepoint'], roll_coins, color='#d62728', linewidth=2.5, label='Coins (3-session mean)')
        
        # Stars
        raw_stars = p_df['stars_mean']
        roll_stars = raw_stars.rolling(window=window, min_periods=1).mean()
        # Size proportional to n_blocks
        sizes = p_df['n_blocks'] * 20
        axes[0].scatter(p_df['timepoint'], raw_stars, s=sizes, color='#ffd700', alpha=0.5, edgecolors='gray', label='Stars (marker size = n_blocks)')
        axes[0].plot(p_df['timepoint'], roll_stars, color='#b8860b', linewidth=2.5, label='Stars (3-session mean)')
        
        axes[0].set_ylabel("Efficiency Mean")
        axes[0].legend(loc='lower right', fontsize='small')
        axes[0].set_title(f"Player {player.upper()} — BEARS Treatment (Timepoint Indexed)")
        
        # --- Panel 2: Naming Accuracy ---
        raw_acc = p_df['acc_mean']
        roll_acc = raw_acc.rolling(window=window, min_periods=1).mean()
        axes[1].plot(p_df['timepoint'], raw_acc, color='#2ca02c', alpha=0.3, marker='D', markersize=4)
        axes[1].plot(p_df['timepoint'], roll_acc, color='#2ca02c', linewidth=2.5, label='Accuracy (Rolling Mean)')
        axes[1].set_ylabel("Accuracy")
        axes[1].set_ylim(0, 1.05)
        axes[1].grid(True, linestyle='--', alpha=0.5)
        
        # --- Panel 3: Naming RT ---
        raw_rt = p_df['rt_median']
        roll_rt = raw_rt.rolling(window=window, min_periods=1).mean()
        axes[2].plot(p_df['timepoint'], raw_rt, color='#9467bd', alpha=0.3, marker='^', markersize=4)
        axes[2].plot(p_df['timepoint'], roll_rt, color='#9467bd', linewidth=2.5, label='RT (Rolling Mean)')
        axes[2].set_ylabel("Median RT (sec)")
        axes[2].set_xlabel("Treatment Timepoint")
        
        # Final Polish
        for ax in axes:
            ax.grid(True, linestyle=':', alpha=0.7)
            # Only integer ticks on x-axis
            ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
            
        plt.tight_layout()
        
        # Save
        save_path = f"{output_dir}/player_{player}_clean_dashboard.png"
        plt.savefig(save_path, dpi=150)
        print(f"Saved dashboard for {player} to {save_path}")
        plt.show()
        plt.close()

create_dashboards(summary_data)

## Interpretation Guidance

1. **Timepoint vs. Session**: The x-axis uses `timepoint` to ensure consecutive integers (1, 2, 3...), which is more appropriate for learning curve analysis than raw clinical session IDs which might have gaps.
2. **Marker Sizes**: In the Efficiency panel, the size of the gold stars reflects the volume of practice (number of category blocks completed in that session).
3. **Rolling Means**: The thick lines represent a 3-timepoint rolling average, which helps identify underlying trends despite session-to-session variability.
4. **Accuracy & Speed**: Ideally, as the player progresses, we should see the green line (accuracy) ascending and the purple line (RT) descending.